[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab01_What_Is_AI.ipynb)

In [ ]:
#@title Lab 01 — What Is AI, Really?
print("""
 █████             █████    █████                                                     █████████   █████
░░███             ░░███    ░░███                                         ███         ███░░░░░███ ░░███ 
 ░███  ████████   ███████   ░███         ██████   █████ ███ █████       ░███        ░███    ░███  ░███ 
 ░███ ░░███░░███ ░░░███░    ░███        ░░░░░███ ░░███ ░███░░███     ███████████    ░███████████  ░███ 
 ░███  ░███ ░███   ░███     ░███         ███████  ░███ ░███ ░███    ░░░░░███░░░     ░███░░░░░███  ░███ 
 ░███  ░███ ░███   ░███ ███ ░███      █ ███░░███  ░░███████████         ░███        ░███    ░███  ░███ 
 █████ ████ █████  ░░█████  ███████████░░████████  ░░████░████          ░░░         █████   █████ █████
░░░░░ ░░░░ ░░░░░    ░░░░░  ░░░░░░░░░░░  ░░░░░░░░    ░░░░ ░░░░                      ░░░░░   ░░░░░ ░░░░░ 
                                                                                                       
                                                                                                       
                                                                                                       

   Lab 01 // What Is AI, Really?
   Melbourne Law Masters 2026
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* - [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# What Is AI, Really?

## Exercise Overview

Welcome to International Law and AI, Lab Session 1. In this notebook, you're going to confront a fundamental question: **what does "AI" even mean?**

The term "artificial intelligence" gets thrown around constantly in legal discourse–in discussions of autonomous weapons, algorithmic bias, judicial AI systems, and regulatory frameworks. But practitioners and policymakers often conflate two radically different approaches:

1. **Symbolic AI (Classical / Rule-Based)**: Systems that follow explicit, hardcoded rules. You write the logic, and the machine executes it.
2. **Subsymbolic AI (Machine Learning / Neural Networks)**: Systems that learn statistical patterns from data. No explicit rules–only learned weights and activations.

These are *not* the same thing. They fail differently. They're auditable differently. They're accountable differently. And crucially for lawyers, they're regulated differently.

In this lab, you'll:
- Build a simple rule-based image classifier (symbolic AI)
- Deploy a pre-trained deep neural network (subsymbolic AI)
- Test both systems on straightforward images *and* edge cases (drones, kites, flying fish)
- Watch both fail, but fail in completely different ways
- Reflect on what this means for legal frameworks, transparency, and accountability

By the end, you'll understand why computer scientists, lawyers, and policymakers talk past each other–and how to bridge that gap.

## Where this lab sits on the AI map

Three broad paradigms, one rough map. This course uses all three.

| Paradigm | What it does | How | Example tools | This lab |
| --- | --- | --- | --- | --- |
| **Symbolic** | Follows explicit rules | Humans write logic; machine executes | IF-THEN rules, expert systems, treaty-as-code | Labs 01 (P1), 07 (P1) |
| **Subsymbolic (pattern recognition)** | Learns to classify, detect, or measure similarity | Neural network trained on labelled data; no explicit rules | CNNs (MobileNet, YOLO), BERT embeddings, sentence-transformers | Labs 01 (P2), 04, 06, 07 (P3), 09 (P3–4), 10 (P6) |
| Generative | Produces new text, image, or action | Large model predicts next token from a probability distribution | GPT, Gemini, Claude; agentic systems layer tools on top | Labs 02, 05, 09 (P1–2), 10 (P3–5) |

Generative models are technically subsymbolic too – they are neural networks under the hood. They are separated out because their behaviour (producing new content rather than classifying existing content) poses distinctive legal problems: authorship, attribution, hallucination, autonomy.

**This lab sits in:** **Symbolic** (Part One) and **Subsymbolic** (Part Two).

In [ ]:
#@title Paper notes setup
#@markdown This lab will append your reflections to a markdown file you can download at the end of the session and use as material for your 5,000-word paper.

from pathlib import Path
from datetime import datetime

NOTES_PATH = Path("/content/paper_notes_lab_01.md")
if not NOTES_PATH.exists():
    NOTES_PATH.write_text(
        f"# Lab 01 – paper notes\n\nStarted: {datetime.now().isoformat(timespec='minutes')}\n\n"
    )

def _append_to_notes(heading, body):
    with NOTES_PATH.open("a") as f:
        f.write(f"\n## {heading}\n\n{body}\n")

print(f"Notes will accumulate in: {NOTES_PATH}")
print("At the end of the lab, open the Files panel (folder icon, left) to download.")

# Part One // The Symbolic Approach

In [ ]:
#@title Install Dependencies
!pip install -q Pillow requests torch torchvision
print("✓ Dependencies installed successfully.")

In [ ]:
#@title Import Libraries
import requests
from PIL import Image
from io import BytesIO
import numpy as np
from IPython.display import Image as IPImage, display, Markdown
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded.")

## What is Symbolic AI?

In classical symbolic AI, we write **explicit rules** that map inputs to outputs. The system is completely transparent: you can read the code and understand exactly why it made a decision.

Example rule: "If an object has wings AND is small AND has feathers → it's a bird."

This is how **law** traditionally works–you have explicit rules (statutes, precedents), and you apply them to facts.

The problem? Reality is messier than any set of rules can capture.

In [ ]:
#@title Build a Rule-Based Image Classifier
#@markdown This is a deliberately simple system. It examines basic image properties and applies hardcoded rules.

class SymbolicClassifier:
    """A rule-based classifier using image properties."""
    
    @staticmethod
    def analyze_image(image):
        """Extract simple features from an image."""
        img_array = np.array(image)
        
        height, width = img_array.shape[:2]
        aspect_ratio = width / height if height > 0 else 1
        
        # Check for dominant colors
        if len(img_array.shape) == 3:
            mean_r = np.mean(img_array[:, :, 0])
            mean_g = np.mean(img_array[:, :, 1])
            mean_b = np.mean(img_array[:, :, 2])
        else:
            mean_r = mean_g = mean_b = np.mean(img_array)
        
        # Check for edges (rough indicator of complexity)
        edges = np.abs(np.diff(np.mean(img_array, axis=2), axis=0)).mean()
        
        return {
            'width': width,
            'height': height,
            'aspect_ratio': aspect_ratio,
            'mean_r': mean_r,
            'mean_g': mean_g,
            'mean_b': mean_b,
            'edge_complexity': edges
        }
    
    @staticmethod
    def classify(image):
        """Apply explicit rules to classify the image."""
        features = SymbolicClassifier.analyze_image(image)
        
        ar = features['aspect_ratio']
        r, g, b = features['mean_r'], features['mean_g'], features['mean_b']
        edge = features['edge_complexity']
        
        # Rule 1: Wide, blue-ish → aircraft
        if ar > 2.5 and b > r and b > g:
            return "Aircraft", 0.8
        
        # Rule 2: Square-ish, gray, low complexity → likely a drone
        if 0.8 < ar < 1.2 and abs(r - g) < 20 and abs(g - b) < 20 and edge < 5:
            return "Drone", 0.75
        
        # Rule 3: Tall, brown → likely a bird or kite
        if ar < 1 and r > g > b:
            return "Bird/Kite", 0.6
        
        # Rule 4: Complex edges, varied colors → general object
        if edge > 10:
            return "Complex Object", 0.5
        
        # Default
        return "Unknown", 0.3

print("✓ SymbolicClassifier defined.")

In [ ]:
#@title Load Test Images (Straightforward Cases)
#@markdown We'll test on a few public domain images from Wikimedia Commons.

test_images = {
    'Bird': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3d/Perdix_perdix_2.jpg/800px-Perdix_perdix_2.jpg',
    'Commercial Airliner': 'https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/2017_Airbus_A380_MSN_272.jpg/1024px-2017_Airbus_A380_MSN_272.jpg',
}

loaded_images = {}

for label, url in test_images.items():
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content))
        # Resize for consistency
        img.thumbnail((400, 300))
        loaded_images[label] = img
        print(f"✓ Loaded: {label}")
    except Exception as e:
        print(f"✗ Failed to load {label}: {e}")

print(f"\n✓ Loaded {len(loaded_images)} test images.")

In [ ]:
#@title Test the Symbolic Classifier
#@markdown Let's see how our rule-based system performs on straightforward images.

print("=" * 70)
print("SYMBOLIC (RULE-BASED) CLASSIFIER RESULTS")
print("=" * 70)

for label, img in loaded_images.items():
    prediction, confidence = SymbolicClassifier.classify(img)
    print(f"\n{label}:")
    print(f"  → Classified as: {prediction}")
    print(f"  → Confidence: {confidence:.2%}")
    print(f"  → Size: {img.size}")

print("\n" + "=" * 70)
print("✓ Symbolic classification complete.")

## Key Insight: The Brittleness of Rules

Notice what just happened:

1. **Transparency**: You could read every line of code. You know *exactly* why the classifier made each decision. You could audit it. You could change the rules.
2. **Predictability**: If you understand the rules, you can predict the system's behavior.
3. **Brittleness**: The system works fine on images that fit the rules. But what about a flying fish? A kite? A drone? A bat? One unexpected feature, and the system fails–often confidently and unpredictably.

This is why symbolic AI has largely been superseded by machine learning for classification tasks. **But** it's also why symbolic AI is still used in critical domains like law, medicine, and policy–because explainability and auditability matter.

Now let's compare this to a neural network approach.

# Part Two // The Neural Network

## What is Subsymbolic AI (Neural Networks)?

In contrast to symbolic AI, neural networks don't follow explicit rules. Instead, they learn **statistical patterns** from massive datasets.

A pre-trained model like MobileNetV2 has seen millions of images during training (ImageNet dataset). It has learned to recognize patterns–not through rules that humans wrote, but through mathematical optimization.

The system is a "black box": you can't easily explain *why* it made a particular decision. But it's often much more accurate on diverse, real-world inputs.

This is the double-edged sword: **power without transparency**.

In [ ]:
#@title Load Pre-trained Neural Network
#@markdown We'll use MobileNetV2, a lightweight convolutional neural network trained on ImageNet.

import torch
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2
import json

# Load model and weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = mobilenet_v2(weights='DEFAULT')
model.to(device)
model.eval()

print(f"✓ MobileNetV2 loaded on {device}")
print(f"✓ Pre-trained on ImageNet (1,000 classes)")

# Load ImageNet class labels
imagenet_labels_url = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
try:
    response = requests.get(imagenet_labels_url)
    imagenet_classes = response.text.strip().split('\n')
    print(f"✓ Loaded {len(imagenet_classes)} ImageNet class labels")
except:
    print("✗ Could not load ImageNet labels (will show class indices instead)")
    imagenet_classes = [f"Class {i}" for i in range(1000)]

In [ ]:
#@title Neural Network Classification Function

def classify_with_neural_net(image, top_k=5):
    """Classify image using pre-trained MobileNetV2."""
    
    # Preprocessing pipeline
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    # Convert PIL image to RGB if needed
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Preprocess and get prediction
    input_tensor = preprocess(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
    
    # Get top-k predictions
    probabilities = torch.nn.functional.softmax(output, dim=1)[0]
    top_prob, top_indices = torch.topk(probabilities, k=top_k)
    
    results = []
    for prob, idx in zip(top_prob, top_indices):
        class_label = imagenet_classes[idx.item()]
        results.append((class_label, prob.item()))
    
    return results

print("✓ Neural network classifier function defined.")

In [ ]:
#@title Test the Neural Network on Same Images

print("=" * 70)
print("NEURAL NETWORK (SUBSYMBOLIC) CLASSIFIER RESULTS")
print("=" * 70)

for label, img in loaded_images.items():
    predictions = classify_with_neural_net(img, top_k=5)
    print(f"\n{label}:")
    for i, (class_name, prob) in enumerate(predictions, 1):
        print(f"  {i}. {class_name:<35} {prob:>6.2%}")

print("\n" + "=" * 70)
print("✓ Neural network classification complete.")

## Key Insight: Power vs. Opacity

Notice the contrast:

| Aspect | Symbolic AI | Neural Network |
|--------|------------|----------------|
| **Rules** | Explicit (you read them) | Hidden (learned patterns) |
| **Accuracy** | Often limited | Usually better on real-world data |
| **Explainability** | High (you can audit the rules) | Low (black box) |
| **Reliability** | Predictable, brittle | Powerful, surprising failures |
| **Transparency** | Complete | Minimal |

Now comes the interesting part: **edge cases**. Let's see where both systems break.

# Part Three // Edge Cases – Where Things Break

## The Ambiguous Cases

What happens when we test both systems on images that violate the assumptions built into each?

- **A drone**: Is it a bird? An aircraft? A toy?
- **A kite**: Shaped like a bird, but definitely not one.
- **A flying fish**: Has wings, but isn't a bird.
- **A bat**: Flies like a bird, but it's a mammal.
- **A bird with outstretched wings photographed from below**: Hard to classify confidently.

These are the cases where both systems reveal their fundamental limitations–but in different ways.

Also: this is known as the **XKCD 1425 problem**. See: https://xkcd.com/1425/ ("Tasks")

In [ ]:
#@title Load Edge Case Images
#@markdown These are the tricky ones.

edge_cases = {
    'Drone (DJI-style)': 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e8/DJI_Phantom_3_Quadcopter.jpg/640px-DJI_Phantom_3_Quadcopter.jpg',
    'Kite (Diamond)': 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg',  # Fallback to something else if not available
    'Bat in flight': 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/00/Artibeus_lituratus1.png/440px-Artibeus_lituratus1.png',
}

edge_loaded = {}

for label, url in edge_cases.items():
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content))
        img.thumbnail((400, 300))
        edge_loaded[label] = img
        print(f"✓ Loaded: {label}")
    except Exception as e:
        print(f"✗ Failed to load {label}: {e}")

print(f"\n✓ Loaded {len(edge_loaded)} edge case images.")

In [ ]:
#@title Compare Both Classifiers on Edge Cases

print("=" * 100)
print("EDGE CASE ANALYSIS: SYMBOLIC vs. NEURAL NETWORK")
print("=" * 100)

for label, img in edge_loaded.items():
    print(f"\n{'=' * 100}")
    print(f"TEST CASE: {label}")
    print(f"{'=' * 100}")
    
    # Symbolic result
    sym_pred, sym_conf = SymbolicClassifier.classify(img)
    print(f"\n[SYMBOLIC AI]")
    print(f"  Classification: {sym_pred}")
    print(f"  Confidence: {sym_conf:.2%}")
    print(f"  → This system followed explicit rules.")
    
    # Neural network result
    nn_preds = classify_with_neural_net(img, top_k=5)
    print(f"\n[NEURAL NETWORK]")
    for i, (class_name, prob) in enumerate(nn_preds, 1):
        print(f"  {i}. {class_name:<30} {prob:>6.2%}")
    print(f"  → This system learned statistical patterns.")
    
    print(f"\n[ANALYSIS]")
    print(f"  Did they agree? {sym_pred == nn_preds[0][0]}")
    print(f"  ∴ This is a case where both approaches struggle.")

print(f"\n{'=' * 100}")

## Interactive Test: Try Your Own Image

Now it's your turn. Paste a public image URL below and we'll classify it with both systems. Try to find an edge case that confuses them!

**Tips**:
- Use images from Wikimedia Commons, Pexels, or Pixabay
- Try ambiguous images: animals that don't fit neatly (platypus? okapi?), objects that look like animals (scarecrow? kite?)
- See if you can make both systems fail in interesting ways

In [ ]:
#@title Test Your Own Image
test_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3d/Perdix_perdix_2.jpg/800px-Perdix_perdix_2.jpg" #@param {type:"string"}

try:
    response = requests.get(test_image_url, timeout=5)
    test_img = Image.open(BytesIO(response.content))
    test_img.thumbnail((400, 300))
    
    print("=" * 70)
    print("YOUR IMAGE CLASSIFICATION TEST")
    print("=" * 70)
    
    # Symbolic
    sym_pred, sym_conf = SymbolicClassifier.classify(test_img)
    print(f"\n[SYMBOLIC AI]")
    print(f"  → {sym_pred} ({sym_conf:.2%} confidence)")
    
    # Neural network
    nn_preds = classify_with_neural_net(test_img, top_k=5)
    print(f"\n[NEURAL NETWORK]")
    for i, (class_name, prob) in enumerate(nn_preds, 1):
        print(f"  {i}. {class_name:<30} {prob:>6.2%}")
    
    print(f"\n[COMPARISON]")
    agreement = "YES" if sym_pred in [pred[0] for pred in nn_preds[:3]] else "NO"
    print(f"  Agreement (top-3)? {agreement}")
    
except Exception as e:
    print(f"✗ Error loading image: {e}")
    print(f"  Make sure your URL points to a valid public image.")

## Key Insight: Different Failures

This is crucial:

- **Symbolic AI fails predictably**: If you understand the rules, you can predict exactly where it will fail. A rule like "square-ish + gray → drone" will confidently misclassify any gray square object. You can audit and fix the rule.

- **Neural Network fails mysteriously**: It might confidently classify a kite as "golden eagle" or a drone as "helicopter". You don't know *why*–the learned patterns are opaque. You can't easily audit or fix it. But you might be able to retrain it on more diverse data.

Both fail. But from a legal and regulatory perspective, these failures have very different implications for **accountability**, **transparency**, and **auditability**.

## Side-by-side comparison: symbolic vs neural across all images

Now that you have run both classifiers on both straightforward and edge-case images, the table below shows them side by side. The "truth" column is the photograph's actual subject (annotated when the image was loaded).

The agreement / disagreement columns are not the point. The point is the **failure mode** in each disagreement: when the symbolic classifier is wrong it is wrong because of a hand-written rule you can read; when the neural network is wrong it is wrong because of millions of weights you cannot.

In [ ]:
#@title Build comparison DataFrame

import pandas as pd

rows = []

# Straightforward cases (loaded earlier into `loaded_images`)
for label, img in loaded_images.items():
    sym_pred, sym_conf = SymbolicClassifier.classify(img)
    nn_preds = classify_with_neural_net(img, top_k=1)
    nn_top, nn_score = nn_preds[0]
    rows.append({
        "image": label,
        "truth": label,
        "symbolic_verdict": f"{sym_pred} ({sym_conf:.2f})",
        "neural_top1": f"{nn_top} ({nn_score:.2f})",
        "symbolic_correct": sym_pred.lower() in label.lower() or label.lower() in sym_pred.lower(),
        "neural_correct": nn_top.lower() in label.lower() or label.lower() in nn_top.lower(),
    })

# Edge cases (loaded into `edge_loaded`)
for label, img in edge_loaded.items():
    sym_pred, sym_conf = SymbolicClassifier.classify(img)
    nn_preds = classify_with_neural_net(img, top_k=1)
    nn_top, nn_score = nn_preds[0]
    truth_simple = label.split(" ")[0].lower()
    rows.append({
        "image": label,
        "truth": label,
        "symbolic_verdict": f"{sym_pred} ({sym_conf:.2f})",
        "neural_top1": f"{nn_top} ({nn_score:.2f})",
        "symbolic_correct": truth_simple in sym_pred.lower(),
        "neural_correct": truth_simple in nn_top.lower(),
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))
print()
print(f"Symbolic correct: {int(results_df['symbolic_correct'].sum())}/{len(results_df)}")
print(f"Neural   correct: {int(results_df['neural_correct'].sum())}/{len(results_df)}")
print()
print("Now the prompt: pick a row where the two classifiers disagree.")
print("Write 2 sentences in the cell below capturing what each system 'saw' and")
print("which failure mode would matter more in a legal decision-support context.")

# Make the dataframe available for the reflection cell
globals()["lab01_comparison_df"] = results_df


In [ ]:
#@title Per-row observations (saved to paper notes)
#@markdown Pick the most interesting row in the table above. Write 2 sentences
#@markdown on what each system did and what its failure mode tells you.

row_label = "Drone (DJI-style)"  #@param {type:"string"}
your_observation = ""             #@param {type:"string"}

if your_observation.strip():
    matching_rows = lab01_comparison_df[lab01_comparison_df["image"] == row_label]
    context = matching_rows.to_string(index=False) if not matching_rows.empty else f"(row '{row_label}' not in the table)"
    body = f"Row: {row_label}\n\nClassifier output:\n{context}\n\nObservation: {your_observation}"
    _append_to_notes("Lab 01 row observation", body)
    print(f"Saved to /content/paper_notes_lab_01.md – row '{row_label}' + your observation.")
else:
    print("(blank — type an observation in the field above and re-run this cell)")


# Part Four // Reflection – What Does This Mean for Law?

## Why This Matters for International Law

You've just experienced the fundamental divide in AI: **symbolic (rule-based) vs. subsymbolic (learned patterns)**.

When policymakers, lawyers, and technologists talk about "AI", they often mean completely different things. And this matters enormously.

### Symbolic AI and the Law

Traditional legal reasoning *is* symbolic AI:
- **Statutes**: "If [condition X] then [legal consequence Y]"
- **Case law**: "In case [A], the court ruled [B] because [reason C]"
- **Legal reasoning**: Application of explicit rules to facts

When we build AI systems that use explicit rules (e.g., "if blood alcohol > 0.08% → DUI"), lawyers understand them. We can audit them. We can argue about whether the rules are *just*. We have a framework for challenge and appeal.

### Neural Networks and the "Black Box" Problem

When we deploy machine learning systems (neural networks, decision trees, etc.) in legal/policy contexts, we lose this transparency:
- **No explicit rules to audit**: The system learned patterns from data, but we can't read them as clearly as code
- **Emergent behavior**: The system might discover patterns humans never wrote or intended
- **Accountability gap**: If a system makes a biased decision, we might not know *why* it happened
- **Reproducibility**: Training on slightly different data can produce different results

### Real-World Applications (You'll Study These Later)

This distinction matters in cases you'll encounter in this course:

1. **Autonomous Weapons (Session 6)**: Should a drone have rule-based targeting ("do not fire if civilians present") or learned patterns? Which is more accountable in warfare?

2. **Judicial AI (Session 9)**: Should sentencing recommendations be rule-based ("prior offense + violent crime → 5-10 years") or learned from historical judicial decisions? The latter might perpetuate historical bias.

3. **Regulatory Frameworks (Session 5)**: The EU AI Act treats high-risk AI systems differently. It requires more documentation, testing, and human oversight for systems that can cause serious harm. But which systems are "high-risk"? Rule-based or learned ones?

4. **Algorithmic Accountability**: If a neural network discriminates based on race, how do you challenge it in court? You can't point to a line of code that says "reject if Black". How do you prove it's the system, not random chance?

### The EU AI Act Perspective

The EU AI Act (which you'll study in detail) distinguishes between:
- **Transparency requirements**: Certain AI systems must be "explainable"
- **High-risk categories**: Systems that can seriously harm rights are regulated more strictly
- **Prohibited categories**: Some uses of AI (certain types of surveillance, manipulation) are banned outright

But the Act mostly doesn't distinguish between symbolic and subsymbolic AI. It focuses on *use case* and *risk level*. This creates a tension:
- A rule-based system for facial recognition might be auditable, but still enable mass surveillance
- A neural network for medical diagnosis might be harder to explain, but save lives

### The Deeper Question

Is transparency (knowing the rules) more important than **accuracy** (getting the right answer)?

- **For medicine**: Accuracy might matter more. An accurate neural network diagnosis is better than a transparent but inaccurate rule.
- **For criminal justice**: Transparency and auditability might matter more. A defendant has the right to challenge their conviction and understand the decision-making.
- **For military action**: Both matter, and they conflict. A transparent rule that says "don't fire at civilians" is more accountable than a learned system that might violate it.

This is the central tension you'll navigate throughout this course.

## Your Reflection: Which Would You Trust?

Take a moment and answer the question below. There's no "right" answer–it depends on context. But your reasoning matters.

In [ ]:
#@title Which approach would you trust more for a legal decision?
your_answer = "It depends on the context" #@param ["Symbolic AI (explicit rules, fully auditable)", "Neural Network (learned patterns, better accuracy)", "Neither—they're both problematic", "It depends on the context"]

print("Your answer:", your_answer)
print("\nWhy this matters:")
print("""
This question gets to the heart of AI governance.

If you chose "Symbolic AI":
  → You value transparency and auditability. You're comfortable with less accuracy
    if it means the rules are clear and defensible.

If you chose "Neural Network":
  → You value outcomes. You're willing to accept some opacity if the system
    performs better. But you're gambling with accountability.

If you chose "It depends on context":
  → You're thinking like a lawyer. You understand that the "right" choice
    depends on stakes, rights, and competing values. Medical diagnosis ≠ Criminal sentencing.

If you chose "Neither":
  → You're skeptical. Good. Both approaches have serious limitations.
    The question isn't which is perfect, but which trade-offs we should accept.
""")

## Next Session

Session 2 will dive into **jurisdictional challenges**: How do different legal systems (US, EU, China, international) approach AI regulation? And why can't they just agree on a single framework?

---

**Questions?** Post on the course forum or reach out to Dr Ridi.

In [ ]:
#@title For your paper
#@markdown One open-ended prompt. Your answer saves to a file you can download and use in your 5,000-word paper.

prompt = """Pick one edge-case image. Write 150 words on what the two classifiers' failures reveal about the auditability of rule-based vs learned systems, and which failure mode you would tolerate better in a legal decision-support tool."""
print(prompt)
print()

paper_note = "" #@param {type:"string"}

if paper_note.strip():
    _append_to_notes(f"For the paper – Lab 01", paper_note)
    print(f"\nSaved to /content/paper_notes_lab_01.md – download from the Files panel (folder icon, left).")
else:
    print("\n(blank – nothing saved yet. Type your answer in the field above and re-run this cell.)")